# Session 10: from notebook to script

Session 9 ended with the cleaning written as two functions, `load_messy`
and `clean_plays`. Today they become part of one program that goes from
the raw file to a finished CSV and chart, and that you can run again next
month with one command.

This notebook builds that program one piece at a time, checking each piece
as it goes. The finished script is
`sessions/session-10/demos/genre_pipeline.py`, and by the end you will have
seen every line of it.

In [ ]:
import os
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

## The notebook way

Here is the kind of cell a notebook grows into. It answers one question,
**which genre do I spend most of my listening time on?**, and it is
correct.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

raw = pd.read_csv("data/messy/plays_messy.csv", keep_default_na=False)
df = raw.rename(columns={"minutes played ": "minutes_played"})
df = df.drop_duplicates().copy()
df["genre"] = df["genre"].str.strip().str.title().replace("", pd.NA)
lookup = df.dropna(subset=["genre"]).groupby("artist_name")["genre"].first()
df["genre"] = df["genre"].fillna(df["artist_name"].map(lookup))
minutes = df["minutes_played"].str.replace("min", "").str.replace(" ", "")
df["minutes_played"] = pd.to_numeric(minutes, errors="coerce")
df.groupby("genre")["minutes_played"].sum().sort_values(ascending=False).round(1)

It works. So what is wrong with it? Three things, and none of them show up
today:

* **It only works in this order.** Every line leans on a variable made
  by the line before it. Spread over twelve cells, run them out of order and
  you get a different answer, or a confusing error.
* **The notebook remembers things you deleted.** Run the next two cells.

In [ ]:
threshold = 5

In [ ]:
long_plays = (df["minutes_played"] > threshold).sum()
print(long_plays, "plays longer than", threshold, "minutes")

Now imagine you delete the `threshold = 5` cell. The second cell still runs,
because `threshold` is still sitting in memory. Save, close, come back
tomorrow, and it fails with a `NameError`. **The notebook you saved is not
the notebook you ran.** *Restart and Run All* is the only honest test of a
notebook, and it is worth doing before you believe any number in one.

* **Next month means doing it all again by hand.** The task from your own
  week that you named in session 1 is like this: the same steps, every
  week, on a new file.

A script fixes all three. It always runs top to bottom, in a fresh Python,
from one command. That is the whole idea of today: **explore in a notebook,
run a script.**

## The shape: a pipeline

Almost every data script answers the same four questions, in the same
order. Each one becomes a function.

| Stage | The question | Our function |
|---|---|---|
| load | Where does the data come from? | `load_messy(path)` |
| clean | What has to be fixed before I can trust it? | `clean_plays(df)` |
| analyse | What is the answer? | `summarise_by_genre(plays)` |
| save | Where does the answer go? | `save_csv(...)`, `save_chart(...)` |

Each stage takes what the one before it returned. Nothing reaches back.
That is what makes each piece something you can test on its own, and
replace on its own.

The names matter. Session 3 asked you to name functions after what they
return; this is where that pays off, because `main()` will read like the
table above.

## Stage 1: load

Deliberately boring. It reads the file exactly as it is on disk and
changes nothing, so that if something looks wrong later you know the file
is not to blame.

In [ ]:
MESSY = Path("data/messy/plays_messy.csv")


def load_messy(path):
    """Return the messy CSV at path as a DataFrame, with nothing guessed.

    keep_default_na=False stops pandas turning "NA" and empty cells into
    missing values on its own. Every text column arrives exactly as it was
    written, and clean_plays decides what counts as missing.
    """
    return pd.read_csv(path, keep_default_na=False)


raw = load_messy(MESSY)
assert raw.shape == (2223, 9)
raw.head(3)

That `assert` is the habit for today: write a function, then check it once
with a number you know, before building the next thing on top of it.

## Stage 2: clean

This is your session 9 work, unchanged: `load_messy` above and the cell
below are copied from `sessions/session-09/solutions/cleaning.py`. Copied,
not imported, because a pipeline should stand on its own. If the session 9
folder moved tomorrow, this script should not care.

Nothing in it is new. `fill_from_artist` is the session 4 lookup
dictionary built with `groupby`, and `parse_dates` tries each date format
on purpose, so `05/01/2024` is always the 5th of January.

In [ ]:
# The column order of data/clean/plays.csv.
COLUMNS = [
    "play_id", "played_at", "artist_name", "track_name", "genre",
    "country", "minutes_played", "device", "skipped",
]

# The three ways a date is written in the messy file, tried in this order.
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"]


def parse_dates(text):
    """Return a column of dates written in any of DATE_FORMATS as datetimes.

    Each format is tried on purpose, one at a time. A single blanket parse
    reads 05/01/2024 as the 1st of May, without a word of warning.
    """
    dates = pd.to_datetime(text, format=DATE_FORMATS[0], errors="coerce")
    for date_format in DATE_FORMATS[1:]:
        dates = dates.fillna(pd.to_datetime(text, format=date_format, errors="coerce"))
    assert dates.notna().all(), "some dates are in none of DATE_FORMATS"
    return dates


def fill_from_artist(df, column):
    """Return column with its gaps filled from the same artist's other rows.

    Every artist has exactly one genre and one country, so a play with the
    country missing can borrow it from any other play by the same artist.
    This is the session 4 lookup dictionary, built with groupby.
    """
    lookup = df.dropna(subset=[column]).groupby("artist_name")[column].first()
    return df[column].fillna(df["artist_name"].map(lookup))


def clean_plays(df):
    """Return a cleaned copy of the messy plays, shaped like data/clean/plays.csv.

    The decisions, in order:
      * the "minutes played " column is renamed to minutes_played
      * exact duplicate rows are dropped (40 in the messy file), first,
        because they are exact copies and every later step is then smaller
      * genre is stripped and title-cased, device stripped and lower-cased
      * dates in three formats are parsed deliberately, then written back as
        YYYY-MM-DD text, which is how the clean file stores them
      * " min" and spaces are removed from minutes before converting; a blank
        stays missing (NaN), because nobody knows how long that play was
      * "NA" in country means missing here (no artist is from Namibia), and
        missing genre and country are filled from the artist's other plays
      * a missing device becomes "unknown", so those plays still get counted

    The input DataFrame is not changed.
    """
    df = df.rename(columns={"minutes played ": "minutes_played"})
    df = df.drop_duplicates().copy()
    assert df["play_id"].is_unique, "a play_id appears twice with different values"

    # Work from text in every damaged column, so this behaves the same even if
    # the file was read without keep_default_na=False.
    text_columns = ["played_at", "genre", "country", "device", "minutes_played"]
    df[text_columns] = df[text_columns].fillna("").astype(str)

    df["genre"] = df["genre"].str.strip().str.title().replace("", pd.NA)
    df["device"] = df["device"].str.strip().str.lower().replace("", "unknown")
    df["country"] = df["country"].str.strip().replace(["NA", ""], pd.NA)
    df["genre"] = fill_from_artist(df, "genre")
    df["country"] = fill_from_artist(df, "country")

    df["played_at"] = parse_dates(df["played_at"]).dt.strftime("%Y-%m-%d")

    # Remove " min", and any spaces: the export writes thousands as "1 234.50".
    # No play in this file is that long, but nothing else would catch it.
    minutes = df["minutes_played"].str.replace("min", "").str.replace(" ", "")
    df["minutes_played"] = pd.to_numeric(minutes, errors="coerce")

    # errors="coerce" never complains, so check it only blanked the blanks.
    unreadable = df["minutes_played"].isna() & (minutes != "")
    assert not unreadable.any(), "some minutes could not be read as numbers"

    return df[COLUMNS].sort_values("play_id").reset_index(drop=True)


print(parse_dates(pd.Series(["2024-03-07", "07/03/2024", "07-03-2024"])).tolist())

plays = clean_plays(raw)
assert len(plays) == 2183
plays.head(3)

Three ways of writing the 7th of March, all landing on the same day: a
check small enough to do by eye. Then the real thing: 2,183 plays, the
clean file's count.

The docstring lists the decisions. That is not decoration: in session 9 you
saw that each one changes the answer, and a reader of the script should not
have to work them out from the code. The `assert` lines inside are session
9's too: `errors="coerce"` never complains, so the function checks for
itself that it only blanked the blanks.

Because we have the clean file, we can check the cleaning properly rather
than by eye: same columns, same types, and how many values still differ.

In [ ]:
clean = pd.read_csv("data/clean/plays.csv")

assert list(plays.columns) == list(clean.columns)
assert plays.dtypes.equals(clean.dtypes)

for column in clean.columns:
    differ = (plays[column] != clean[column]) & plays[column].notna()
    missing = plays[column].isna().sum()
    print(f"{column:15} {differ.sum():>3} different   {missing:>3} missing")

Genre, country and every date match the clean file exactly. What is left is
what cleaning cannot invent: **69 devices** we had to call `"unknown"`, and
**94 plays** whose minutes were blank. Those 94 are still plays, so they
count in the play totals, but they add nothing to the minutes. Hold on to
that; it comes back at the end of this notebook.

## Stage 3: analyse, in the session 3 shape

In session 3 you wrote this, and the rule that went with it was **functions
that calculate return; only the outer layer prints**:

```python
def total(numbers):
    """Return the sum of a list of numbers."""
    running = 0
    for number in numbers:
        running += number
    return running
```

The analysis is the same shape. It takes data, returns an answer, and
prints nothing. Session 3 promised that these shapes would get called by a
pipeline that writes a CSV instead of printing. This is that pipeline.

In [ ]:
def summarise_by_genre(plays):
    """Return one row per genre: plays, minutes, average minutes, skip rate."""
    summary = (plays.groupby("genre")
               .agg(plays=("play_id", "count"),
                    minutes=("minutes_played", "sum"),
                    average_minutes=("minutes_played", "mean"),
                    skip_rate=("skipped", "mean"))
               .sort_values("minutes", ascending=False)
               .reset_index())
    summary["skip_rate"] = summary["skip_rate"] * 100
    return summary.round({"minutes": 1, "average_minutes": 2, "skip_rate": 1})


summary = summarise_by_genre(plays)
assert summary["plays"].sum() == 2183
summary

Because it returns a table instead of printing one, the same function can
feed a CSV, a chart, a check, or another notebook, and none of them need it
to change.

## Stage 4: save

Two functions, one per output. Both take the summary and a path, and both
make the folder if it is missing, so the first run on a new machine does
not fail.

In [ ]:
OUTPUT = Path("output")


def save_csv(summary, path):
    """Write the summary table to path as a CSV, making the folder if needed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(path, index=False)


def save_chart(summary, path):
    """Draw minutes per genre as a bar chart, titled with the answer, to path."""
    top = summary.iloc[0]
    share = top["minutes"] / summary["minutes"].sum() * 100

    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.barh(summary["genre"], summary["minutes"], color="#4f6ddb")
    ax.invert_yaxis()                                # biggest at the top
    ax.bar_label(bars, labels=[f"{m:,.1f}" for m in summary["minutes"]], padding=3)
    ax.set_title(f"{top['genre']} gets {share:.1f}% of my listening time")
    ax.set_xlabel("minutes played, 2024 to 2025")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


save_csv(summary, OUTPUT / "genre_summary.csv")
save_chart(summary, OUTPUT / "genre_minutes.png")
print((OUTPUT / "genre_summary.csv").exists(), (OUTPUT / "genre_minutes.png").exists())

`plt.close(fig)` instead of `plt.show()`: a script has nobody to show the
chart to, so it saves the file and lets the figure go. To look at it here,
open the PNG it wrote:

In [ ]:
from IPython.display import Image

Image(filename="output/genre_minutes.png", width=640)

The title states the answer, with the share to one decimal using the `:.1f`
from session 1. A title like "Minutes per genre" would make the reader work
out the answer; this one tells them.

## The check: read it back

In the session 4 stretch you wrote a file, read it back, and checked the
totals against the data it came from. That kind of check is what tells
you whether a run of a pipeline worked, so it gets a function too.

In [ ]:
def check_summary(path, plays):
    """Read the written CSV back and confirm it agrees with the cleaned plays."""
    back = pd.read_csv(path)
    file_minutes = back["minutes"].sum()
    data_minutes = plays["minutes_played"].sum()

    assert back["plays"].sum() == len(plays), "plays went missing on the way"
    # Each genre was rounded to one decimal, so allow a little difference.
    assert abs(file_minutes - data_minutes) < 1, "the minutes do not add up"


check_summary(OUTPUT / "genre_summary.csv", plays)
print("the file agrees with the data")

It returns nothing when all is well and stops the run when it is not. A
pipeline that silently writes a wrong file is worse than one that crashes,
because nobody goes looking.

## `main()`: the outer layer

One function calls the stages in order. It is the only one that prints.
Read it out loud and it is the table from the top of this notebook.

In [ ]:
def main():
    """Run the whole pipeline: the outer layer, and the only part that prints."""
    raw = load_messy(MESSY)
    plays = clean_plays(raw)
    summary = summarise_by_genre(plays)

    csv_path = OUTPUT / "genre_summary.csv"
    png_path = OUTPUT / "genre_minutes.png"
    save_csv(summary, csv_path)
    save_chart(summary, png_path)
    check_summary(csv_path, plays)

    print(f"loaded  {len(raw):>5} rows from {MESSY.name}")
    print(f"cleaned {len(plays):>5} rows")
    print(f"top genre: {summary.iloc[0]['genre']}, {summary.iloc[0]['minutes']:,.1f} minutes")


main()

## From notebook to file

Copy every function above into one `.py` file, put the imports at the top,
and add two lines at the bottom:

```python
if __name__ == "__main__":
    main()
```

That file exists: `sessions/session-10/demos/genre_pipeline.py`. Open it next
to this notebook. The functions are the same ones, in the same order.

## What `__name__` is

Python gives every file it loads a variable called `__name__`. When you run
a file, it is `"__main__"`. When another file imports it, it is the file's
own name. A notebook counts as being run:

In [ ]:
print(__name__)

Now import the script instead of running it. Watch what does **not**
happen.

In [ ]:
import sys

sys.path.insert(0, "sessions/session-10/demos")    # where Python looks for imports

import genre_pipeline

print("imported, and its __name__ is", repr(genre_pipeline.__name__))

Nothing ran. No "loaded 2223 rows", no files written. Inside the script,
`__name__` was `"genre_pipeline"`, so the `if` was false and `main()` was
never called. But all its functions are here to use:

In [ ]:
genre_pipeline.summarise_by_genre(plays).head(3)

That is what the two lines buy you: **the same file is a program when you
run it and a toolbox when you import it.** Your project notebook will
import your pipeline exactly like this, so exploring and running use the
same code.

## Paths that work from any folder

Every path in this notebook is relative, like `data/messy/plays_messy.csv`.
A relative path is counted from the folder Python is running in. That is
why every notebook here starts with a cell that walks up to the
repository root, and why session 4 asked you to always run scripts from
there.

A script can do better. Inside a running script, `__file__` is the
location of the script itself, and `pathlib` can build every other path
from it:

```python
ROOT = Path(__file__).resolve().parents[3]
MESSY = ROOT / "data" / "messy" / "plays_messy.csv"
```

A notebook is not a file being run, so `__file__` does not exist here
(uncomment to see). Instead, here is the same arithmetic on the script's
path:

In [ ]:
# print(__file__)
# NameError: name '__file__' is not defined

script = Path("sessions/session-10/demos/genre_pipeline.py").resolve()

print(script.name)                 # genre_pipeline.py
print(script.parents[0].name)      # demos        the folder it is in
print(script.parents[1].name)      # session-10
print(script.parents[2].name)      # sessions
print(script.parents[3] == Path.cwd().resolve())   # True: the repository root

`.resolve()` turns the path into a full one first, so `parents` has
something to count. The `/` between pieces joins them with the right
separator for your computer, backslashes on Windows and forward slashes on
a Mac, so you never type either.

Now the proof. Run the script from three different folders, the way you
would in a terminal with `cd`:

In [ ]:
import subprocess

for folder in [".", "sessions/session-10", "data/clean"]:
    result = subprocess.run([sys.executable, str(script)], cwd=folder,
                            capture_output=True, text=True, check=True)
    print(f"{folder:22} -> {result.stdout.splitlines()[-1]}")

Same answer from every folder. The session 4 version of this script would
have failed in two of the three with a `FileNotFoundError`.

## One last check: what did the missing minutes cost?

The pipeline says 8,608.0 minutes in total. The database from sessions 6
and 7 holds the same log, undamaged:

In [ ]:
import sqlite3

con = sqlite3.connect("data/music.db")
database_total = con.execute("SELECT SUM(minutes_played) FROM plays").fetchone()[0]
con.close()

pipeline_total = plays["minutes_played"].sum()
lost = clean.loc[plays["minutes_played"].isna(), "minutes_played"].sum()

print(f"database:  {database_total:,.1f} minutes")
print(f"pipeline:  {pipeline_total:,.1f} minutes")
print(f"gap:       {database_total - pipeline_total:,.1f} minutes")
print(f"the 94 plays with blank minutes were worth {lost:,.1f} minutes")
assert round(database_total - pipeline_total, 1) == round(lost, 1) == 372.4

The gap is exactly the minutes of the 94 blank plays. So the pipeline is
not wrong; it is honest about what the messy file lost. That is a sentence
worth putting in the README of any project: what the data could not tell
you, and how big that gap is.

Two independent routes to the same number is the best kind of check there
is. You did it in session 4 with a CSV, in session 7 with fan-out, and now
with a whole pipeline.

## Next

`02-device-pipeline-exercise.ipynb`: a notebook-shaped script for a
different question, which you turn into a pipeline.

Virtual environments are terminal work, not notebook work: they are on the
slides, and step by step in `project/README.md`.